# 12 · Apêndice: o raio-X é a tomografia achatada

Exemplos de **tomografia** e **ressonância** do curso de segmentação. Lá, o número do pixel tem unidade
física e a máscara sai de um limiar fixo: na tomografia, "pulmão é de −950 a −400 Hounsfield" vale em
qualquer exame. No nosso raio-X o pixel é um cinza de 0 a 255 sem unidade. Este apêndice usa a mesma
tomografia do curso (CTChest, 3D Slicer) para montar um **raio-X sintético** e perguntar o que a base
do NIH não deixa perguntar: **qual é a largura verdadeira do tórax**, o denominador do ICT?

1. **a tomografia**: Hounsfield, janelas e o pulmão por densidade, corte a corte e em 3D;
2. **a soma**: o raio-X sintético (DRR), somando a atenuação de frente para trás;
3. **um gabarito físico**: o pulmão projetado dá a largura do tórax em milímetros;
4. **o mesmo tórax, cinco revelados**: o que os métodos do projeto medem quando só muda o
   processamento da imagem;
5. **o coração**, que a densidade não separa.

É um apêndice: um paciente só, sem cardiomegalia conhecida. Serve para entender as medidas, não para
avaliá-las em números.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import warnings

import matplotlib.pyplot as plt
import nrrd
import numpy as np
import pandas as pd

from src import config
from src.data.dados import baixar_base, baixar_ct, carregar_imagens, dividir, listar_imagens
from src.model import pretreinada
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import filtros, ict, segmentacao, tomografia

configurar_estilo()
warnings.filterwarnings("ignore", message=".*normaliz.*")
SECAO = "12-raio-x-sintetico"
PARAMS_ICT = dict(sigma_vertical=3, linhas=(50, 90), quantil=75)

volume, cabecalho = nrrd.read(str(baixar_ct()))
volume = volume.astype(np.float32)
espacamento = tuple(float(v) for v in np.diag(cabecalho["space directions"]))
print(f"{volume.shape} voxels de {espacamento[0]:.2f} × {espacamento[1]:.2f} × {espacamento[2]:.2f} mm, "
      f"orientação {cabecalho['space']}")
print(f"Hounsfield de {volume.min():.0f} a {volume.max():.0f}")

## 1. A tomografia: o pixel tem unidade

Cada voxel é densidade em **Hounsfield**: ar ≈ −1000, pulmão perto de −800, água 0, músculo e sangue
+40, osso acima de +300. O olho não lê de −1000 a +3000 de uma vez: a **janela** escolhe uma faixa e a
espalha nos cinzas. A janela de pulmão mostra os vasos; a de mediastino, o coração e os grandes vasos.

O pulmão sai por **densidade**: ar (−950 a −400) dentro do corpo, sem o ar de fora, que encosta na
borda da imagem. Corte a corte, o gás do estômago também passa nessa conta; em 3D ele fica num pedaço
separado que não sobe até o alto do tórax, e sai.

In [ ]:
pulmao_3d = tomografia.pulmao_no_volume(volume)
area_por_corte = pulmao_3d.sum(axis=(0, 1))
z_meio = int(np.argmax(area_por_corte))
cortes = [z_meio + 20, z_meio, z_meio - 25]

fig, eixos = plt.subplots(2, 3, figsize=(13, 8.6))
for ax, (titulo, centro, largura) in zip(eixos[0], [("janela de pulmão", -600, 1500),
                                                    ("janela de mediastino", 40, 400),
                                                    ("janela de osso", 400, 1800)]):
    ax.imshow(tomografia.janela(volume[:, :, z_meio].T, centro, largura), cmap="gray")
    ax.set_title(f"{titulo} (centro {centro}, largura {largura})", fontsize=9)
for ax, z in zip(eixos[1], cortes):
    corte = volume[:, :, z].T
    cor = plt.cm.gray(tomografia.janela(corte, -600, 1500))[..., :3]
    so_no_corte = tomografia.pulmao_no_corte(volume[:, :, z]).T & ~pulmao_3d[:, :, z].T
    cor[pulmao_3d[:, :, z].T] = 0.55 * cor[pulmao_3d[:, :, z].T] + 0.45 * np.array([0.2, 0.85, 0.35])
    cor[so_no_corte] = 0.4 * cor[so_no_corte] + 0.6 * np.array([0.95, 0.4, 0.3])
    ax.imshow(cor)
    ax.set_title(f"corte {z}", fontsize=9)
for ax in eixos.ravel():
    ax.axis("off")
fig.text(0.5, 0.49, "embaixo: pulmão por densidade (verde) e o ar que sai na conta em 3D (vermelho)",
         ha="center", fontsize=10, color=config.AZUL_PROFUNDO)
salvar_figura(fig, SECAO, "tomografia-janelas-e-pulmao")
plt.show()

gas = sum(int((tomografia.pulmao_no_corte(volume[:, :, z]) & ~pulmao_3d[:, :, z]).sum()) for z in range(volume.shape[2]))
print(f"Pulmão: {pulmao_3d.sum() * np.prod(espacamento) / 1e6:.2f} litros · ar fora dos pulmões descartado em 3D: "
      f"{gas * np.prod(espacamento) / 1e3:.0f} mL")

## 2. A soma: o raio-X sintético

O feixe de raio-X atravessa o corpo e perde força conforme a **atenuação** de cada tecido no caminho.
Relativa à água, ela é μ = 1 + HU/1000 (ar 0, água 1, osso ≈ 2). Somando μ de frente para trás, voxel
a voxel, sai a imagem que o filme veria: um **DRR** (*digitally reconstructed radiograph*). Ela fica na
convenção radiológica (lado direito do paciente à esquerda de quem olha) e é levada a 128 × 128, como a
base, com ar dos lados: o paciente fez a tomografia de braços erguidos.

Ao lado, um raio-X verdadeiro da base para comparar.

In [ ]:
soma = tomografia.projetar(tomografia.atenuacao(volume), espacamento)
sintetico, mm_por_px = tomografia.para_128(soma, espacamento[2], espacamento[0])
print(f"Raio-X sintético: 128 × 128, {mm_por_px:.2f} mm por pixel na horizontal")

baixar_base()
catalogo = dividir(listar_imagens())
real = carregar_imagens([catalogo.set_index("id").loc["test/false/138", "caminho"]])[0]

fig, eixos = plt.subplots(1, 3, figsize=(13, 4.4))
eixos[0].imshow(tomografia.janela(volume[:, volume.shape[1] // 2, :].T[::-1], -600, 1500), cmap="gray",
                aspect=espacamento[2] / espacamento[0])
eixos[0].set_title("um corte coronal da tomografia", fontsize=9)
eixos[1].imshow(tomografia.para_uint8(sintetico), cmap="gray")
eixos[1].set_title("a soma de todos os cortes (raio-X sintético)", fontsize=9)
eixos[2].imshow(real, cmap="gray")
eixos[2].set_title("um raio-X real da base (test/false/138)", fontsize=9)
for ax in eixos:
    ax.axis("off")
salvar_figura(fig, SECAO, "corte-soma-e-real")
plt.show()

## 3. Um gabarito físico: a largura do tórax

O pulmão marcado por densidade, projetado do mesmo jeito, mostra onde há pulmão em algum ponto do
caminho do feixe. Não é a "máscara do pulmão" que a PSPNet ou um radiologista desenham no raio-X: os
pulmões passam **por trás e pela frente do coração**, e a projeção cobre o mediastino. Por isso ela
não serve de gabarito de área (Dice). Mas a **borda externa** dela é exatamente a largura interna do
tórax, o **denominador do ICT**, e em milímetros.

In [ ]:
pulmao_proj = tomografia.projetar(pulmao_3d.astype(np.float32), espacamento) > 0
gabarito, _ = tomografia.para_128(pulmao_proj.astype(np.float32), espacamento[2], espacamento[0], interpolacao=0)
gabarito = gabarito > 0.5
torax_px = segmentacao.largura_maxima(gabarito)
print(f"Largura verdadeira do tórax: {torax_px} px = {torax_px * mm_por_px / 10:.1f} cm")

fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.imshow(tomografia.para_uint8(sintetico), cmap="gray")
ax.contour(gabarito.astype(float), levels=[0.5], colors=["#2ECC71"], linewidths=1.6)
linha = int(np.argmax([np.ptp(np.flatnonzero(r)) if r.any() else 0 for r in gabarito]))
x = np.flatnonzero(gabarito[linha])
ax.plot([x.min(), x.max()], [linha, linha], color=config.CORAL, lw=2)
ax.set_title(f"pulmão projetado e a largura do tórax ({torax_px * mm_por_px / 10:.1f} cm)", fontsize=9)
ax.axis("off")
salvar_figura(fig, SECAO, "gabarito-fisico")
plt.show()

## 4. O mesmo tórax, cinco revelados

O aparelho de raio-X não entrega a soma crua: ele "revela" a imagem com curvas de contraste e realce
de borda, e cada fabricante faz de um jeito. No sintético dá para variar só isso, com o tórax físico
fixo:

* **linear**: a soma, só reescalada;
* **gama 1,5**: escurece os cinzas médios (o pulmão fica mais escuro);
* **CLAHE**: o pré-processamento do notebook 02;
* **nitidez suave** e **nitidez forte**: tiram parte do fundo borrado, como o realce dos aparelhos.

E três medidores da largura do tórax: os **pulmões por limiar e morfologia** (Otsu, notebook 04), o
**perfil por linha** do ICT (notebook 04) e os **pulmões da PSPNet** (notebook 10).

In [ ]:
revelados = {
    "linear": tomografia.para_uint8(sintetico),
    "gama 1,5": tomografia.para_uint8(sintetico, gama=1.5),
    "CLAHE": filtros.preprocessar(tomografia.para_uint8(sintetico)),
    "nitidez suave": tomografia.para_uint8(tomografia.nitidez(sintetico, 8, 0.5)),
    "nitidez forte": tomografia.para_uint8(tomografia.nitidez(sintetico, 8, 0.8)),
}
prob = pretreinada.segmentar(np.stack(list(revelados.values())))

linhas, mascaras = [], {}
for (nome, im), p in zip(revelados.items(), prob):
    otsu = ict.segmentar_pulmoes(im) > 0
    perfil = ict.medir_ict(im, **PARAMS_ICT)
    psp = (p[1] > 0.5) | (p[2] > 0.5)
    mascaras[nome] = {"Otsu": otsu, "PSPNet": psp}
    larguras = {"limiar e morfologia": segmentacao.largura_maxima(otsu) or np.nan,
                "perfil do ICT": perfil.torax_px if perfil.ok else np.nan,
                "PSPNet": segmentacao.largura_maxima(psp) or np.nan}
    for metodo, w in larguras.items():
        linhas.append({"revelado": nome, "metodo": metodo, "largura_cm": w * mm_por_px / 10,
                       "erro_pct": 100 * (w - torax_px) / torax_px})
torax = pd.DataFrame(linhas)
salvar_tabela(torax, SECAO, "largura-do-torax", indice=False)
print(f"Verdade: {torax_px * mm_por_px / 10:.1f} cm")
torax.pivot(index="revelado", columns="metodo", values="erro_pct").loc[list(revelados)].round(0)

In [ ]:
fig, eixos = plt.subplots(2, 5, figsize=(16, 7))
for j, (nome, im) in enumerate(revelados.items()):
    for i, metodo in enumerate(["Otsu", "PSPNet"]):
        ax = eixos[i, j]
        ax.imshow(im, cmap="gray")
        ax.contour(gabarito.astype(float), levels=[0.5], colors=["#2ECC71"], linewidths=1.0)
        m = mascaras[nome][metodo]
        if m.any():
            ax.contour(m.astype(float), levels=[0.5], colors=["#E74C3C"], linewidths=1.2)
        nome_metodo = "limiar e morfologia" if metodo == "Otsu" else "PSPNet"
        erro = torax.loc[(torax["revelado"] == nome) & (torax["metodo"] == nome_metodo), "erro_pct"].iloc[0]
        ax.set_title(f"{nome} · {metodo}\n" + ("não achou" if np.isnan(erro) else f"largura {erro:+.0f} %"), fontsize=8)
        ax.axis("off")
fig.suptitle("Verde = pulmão projetado (verdade física) · vermelho = pulmões achados no raio-X",
             fontweight="bold", color=config.AZUL_PROFUNDO)
salvar_figura(fig, SECAO, "cinco-revelados")
plt.show()

## 5. O coração, que a densidade não separa

Coração, sangue e músculo têm todos ≈ +40 HU: nenhum limiar de Hounsfield recorta o coração, e por isso
o apêndice não tem gabarito para o numerador do ICT. O que dá para ver é o que a PSPNet acha no
sintético, comparado com o tamanho que ela costuma achar nos raio-X reais.

In [ ]:
coracao = {nome: segmentacao.maior_componente(p[0] > 0.5) for nome, p in zip(revelados, prob)}
reais = pretreinada.segmentar_com_cache(catalogo["id"].to_numpy(), carregar_imagens(catalogo["caminho"]))
larg_reais = np.array([segmentacao.largura_maxima(segmentacao.maior_componente(p > 0.5)) for p in reais[:, 0]])
print(f"Largura do coração da PSPNet nos raio-X reais: mediana {np.median(larg_reais[larg_reais > 0]):.0f} px")
for nome, m in coracao.items():
    print(f"   sintético, {nome:14s}: {segmentacao.largura_maxima(m):3d} px")

fig, eixos = plt.subplots(1, 5, figsize=(16, 3.6))
for ax, (nome, im) in zip(eixos, revelados.items()):
    ax.imshow(im, cmap="gray")
    if coracao[nome].any():
        ax.contour(coracao[nome].astype(float), levels=[0.5], colors=["#E74C3C"], linewidths=1.4)
    ax.set_title(f"{nome} · coração da PSPNet", fontsize=8)
    ax.axis("off")
salvar_figura(fig, SECAO, "coracao-no-sintetico")
plt.show()

## Conclusões

* **Na tomografia, o limiar fixo funciona porque o pixel tem unidade.** O pulmão sai inteiro só com
  "ar dentro do corpo" (6,2 litros), e a conta em 3D descarta os 52 mL de gás do abdome que, corte a
  corte, passariam por pulmão. É a mesma receita dos notebooks 04 e 08 (limiar, abertura,
  preenchimento, componente, nada encostado na borda), com a vantagem de um limiar que não muda de
  exame para exame.
* **O raio-X é mesmo a soma da tomografia, mas o sintético não é igual ao real.** Pulmões, coluna,
  diafragma e costelas aparecem no lugar certo. A parede lateral sai escura, porque ali o feixe
  atravessa pouco tecido, e o coração quase não se destaca. Faltam as diferenças de um raio-X de
  verdade: o feixe em cone, a radiação espalhada, a postura (deitado e de braços erguidos na tomografia,
  em pé no raio-X) e, principalmente, o processamento do aparelho.
* **A largura do tórax tem um gabarito físico: 29,9 cm.** O pulmão projetado passa por trás do coração,
  então não serve de gabarito de área. A borda externa dele, porém, é a largura interna do tórax.
* **O mesmo tórax, cinco revelados, números diferentes.** A PSPNet acerta a largura com erro de −3 % a
  −8 % no revelado linear, no CLAHE e na nitidez suave, mas não acha os pulmões com gama 1,5 e erra
  −57 % com nitidez forte. Os pulmões por Otsu só aparecem com o CLAHE (−4 %). O perfil do ICT mede o
  corpo, não o tórax (+41 % com CLAHE, −14 % com nitidez forte) ou não mede. Os métodos do projeto
  foram ajustados ao processamento das imagens do NIH; fora dele, o pixel sem unidade cobra o preço.
* **Isso ajuda a ler o ICT do projeto.** A AUC compara exames da mesma base e do mesmo processamento,
  e aguenta um erro sistemático na largura. O corte clínico de 0,5, um valor absoluto, não aguenta: é
  uma explicação possível para a sensibilidade de só 54 % nesse corte (notebook 04).
* **O coração não sai por densidade.** Sangue e músculo têm os mesmos ≈ 40 HU: na janela de
  mediastino o coração tem o cinza da parede do tórax. No sintético, a PSPNet acha um coração de 0 a
  21 px, contra 42 px de mediana nos raio-X reais. Sem gabarito físico para o numerador, o apêndice
  mede só o denominador do ICT.
* **O limite:** um paciente só, sem cardiomegalia conhecida. O apêndice explica as medidas; não as
  avalia em números.